# Olist Brazilian E-Commerce Lakeflow Pipeline

This notebook contains Python Lakeflow pipeline definitions for Bronze, Silver, and Gold tables. Upload or import it into Databricks if you want an `.ipynb` version. For a Lakeflow transformation file, Databricks usually expects a `.py` file, but this notebook keeps the same code in notebook format.

In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import (
    col,
    to_timestamp,
    datediff,
    when,
    round,
    countDistinct,
    sum,
    avg,
    date_format
)

# ============================================================
# Olist Brazilian E-Commerce Lakeflow Pipeline
# Bronze -> Silver -> Gold
#
# Before running this pipeline, make sure:
# 1. CSV files are uploaded to:
#    /Volumes/olist_project/raw/olist_volume/
# 2. Pipeline settings are:
#    Default catalog: olist_project
#    Default schema: pipeline_demo
# ============================================================

BASE_PATH = "/Volumes/olist_project/raw/olist_volume/"


# ============================================================
# BRONZE LAYER
# Raw CSV files loaded from Databricks Volume
# ============================================================

@dp.table(
    name="bronze_orders",
    comment="Raw Olist orders data loaded from CSV file"
)
def bronze_orders():
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(BASE_PATH + "olist_orders_dataset.csv")
    )


@dp.table(
    name="bronze_customers",
    comment="Raw Olist customers data loaded from CSV file"
)
def bronze_customers():
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(BASE_PATH + "olist_customers_dataset.csv")
    )


@dp.table(
    name="bronze_order_items",
    comment="Raw Olist order items data loaded from CSV file"
)
def bronze_order_items():
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(BASE_PATH + "olist_order_items_dataset.csv")
    )


# ============================================================
# SILVER LAYER
# Cleaned and joined data
# ============================================================

@dp.table(
    name="silver_orders_customers_items_clean",
    comment="Cleaned and joined Olist orders, customers, and order items"
)
@dp.expect_or_drop(
    "valid_delivery_date",
    "order_delivered_customer_date IS NOT NULL"
)
@dp.expect_or_drop(
    "valid_estimated_delivery_date",
    "order_estimated_delivery_date IS NOT NULL"
)
@dp.expect_or_drop(
    "valid_price",
    "price IS NOT NULL"
)
@dp.expect_or_drop(
    "valid_freight",
    "freight_value IS NOT NULL"
)
def silver_orders_customers_items_clean():
    orders = spark.read.table("bronze_orders")
    customers = spark.read.table("bronze_customers")
    items = spark.read.table("bronze_order_items")

    joined_df = (
        orders
        .join(customers, "customer_id", "left")
        .join(items, "order_id", "left")
        .filter(col("order_status") == "delivered")
    )

    return (
        joined_df
        .withColumn("order_purchase_timestamp", to_timestamp("order_purchase_timestamp"))
        .withColumn("order_approved_at", to_timestamp("order_approved_at"))
        .withColumn("order_delivered_carrier_date", to_timestamp("order_delivered_carrier_date"))
        .withColumn("order_delivered_customer_date", to_timestamp("order_delivered_customer_date"))
        .withColumn("order_estimated_delivery_date", to_timestamp("order_estimated_delivery_date"))
        .withColumn("shipping_limit_date", to_timestamp("shipping_limit_date"))
        .withColumn("total_item_value", round(col("price") + col("freight_value"), 2))
        .withColumn(
            "delivery_days",
            datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp"))
        )
        .withColumn(
            "delivery_status",
            when(
                col("order_delivered_customer_date") > col("order_estimated_delivery_date"),
                "Late"
            ).otherwise("On time")
        )
    )


# ============================================================
# GOLD LAYER 1
# Delivery and sales by state
# ============================================================

@dp.table(
    name="gold_delivery_sales_by_state",
    comment="Sales and delivery performance by customer state and delivery status"
)
def gold_delivery_sales_by_state():
    silver_df = spark.read.table("silver_orders_customers_items_clean")

    return (
        silver_df
        .groupBy("customer_state", "delivery_status")
        .agg(
            countDistinct("order_id").alias("total_orders"),
            round(sum("total_item_value"), 2).alias("total_revenue"),
            round(avg("delivery_days"), 2).alias("avg_delivery_days"),
            round(avg("freight_value"), 2).alias("avg_freight_value")
        )
    )


# ============================================================
# GOLD LAYER 2
# Monthly sales trend
# ============================================================

@dp.table(
    name="gold_monthly_sales_trend",
    comment="Monthly order and revenue trend"
)
def gold_monthly_sales_trend():
    silver_df = spark.read.table("silver_orders_customers_items_clean")

    return (
        silver_df
        .withColumn("order_month", date_format(col("order_purchase_timestamp"), "yyyy-MM"))
        .groupBy("order_month")
        .agg(
            countDistinct("order_id").alias("total_orders"),
            round(sum("total_item_value"), 2).alias("total_revenue"),
            round(avg("total_item_value"), 2).alias("avg_item_value"),
            round(avg("freight_value"), 2).alias("avg_freight_value")
        )
    )


# ============================================================
# GOLD LAYER 3
# Late delivery rate by state
# ============================================================

@dp.table(
    name="gold_late_delivery_by_state",
    comment="Late delivery percentage by customer state"
)
def gold_late_delivery_by_state():
    silver_df = spark.read.table("silver_orders_customers_items_clean")

    return (
        silver_df
        .groupBy("customer_state")
        .agg(
            countDistinct("order_id").alias("total_orders"),
            countDistinct(
                when(col("delivery_status") == "Late", col("order_id"))
            ).alias("late_orders"),
            round(avg("delivery_days"), 2).alias("avg_delivery_days")
        )
        .withColumn(
            "late_delivery_percentage",
            round((col("late_orders") * 100.0) / col("total_orders"), 2)
        )
    )


# ============================================================
# GOLD LAYER 4
# Freight cost by state
# ============================================================

@dp.table(
    name="gold_freight_by_state",
    comment="Freight cost and revenue comparison by customer state"
)
def gold_freight_by_state():
    silver_df = spark.read.table("silver_orders_customers_items_clean")

    return (
        silver_df
        .groupBy("customer_state")
        .agg(
            countDistinct("order_id").alias("total_orders"),
            round(sum("freight_value"), 2).alias("total_freight_cost"),
            round(avg("freight_value"), 2).alias("avg_freight_value"),
            round(sum("total_item_value"), 2).alias("total_revenue"),
            round(avg(col("freight_value") / col("total_item_value")), 4).alias("avg_freight_to_value_ratio")
        )
    )


# ============================================================
# GOLD LAYER 5
# Seller performance summary
# ============================================================

@dp.table(
    name="gold_seller_performance_summary",
    comment="Seller-level revenue, order count, and delivery performance"
)
def gold_seller_performance_summary():
    silver_df = spark.read.table("silver_orders_customers_items_clean")

    return (
        silver_df
        .groupBy("seller_id")
        .agg(
            countDistinct("order_id").alias("total_orders"),
            countDistinct("product_id").alias("unique_products_sold"),
            round(sum("total_item_value"), 2).alias("total_revenue"),
            round(avg("total_item_value"), 2).alias("avg_item_value"),
            round(avg("delivery_days"), 2).alias("avg_delivery_days")
        )
    )
